In [1]:
import scanpy as sc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [10]:
adata_raw = sc.read_h5ad('/local/hailey/PLX/combined_data.h5ad')#raw data
meta = pd.read_csv('/local/hailey/PLX/combined_data_process_meta_allenmap.csv', index_col=0)
adata_raw.obs = meta
adata_raw

AnnData object with n_obs × n_vars = 2381759 × 49929
    obs: 'dnbCount', 'area', 'nCount_Spatial', 'nFeature_Spatial', 'percent.mito', 'orig.ident', 'x', 'y', 'Sample', 'ChipID', 'Tissue', 'MouseID', 'Group', 'Sex', 'SlicingDate', 'batch', 'clusters', 'predicted_labels', 'over_clustering', 'majority_voting', 'conf_score', 'cell_id', 'class_label', 'class_name', 'class_bootstrapping_probability', 'subclass_label', 'subclass_name', 'subclass_bootstrapping_probability', 'supertype_label', 'supertype_name', 'supertype_bootstrapping_probability', 'cluster_label', 'cluster_name', 'cluster_alias', 'cluster_bootstrapping_probability', 'name', 'Group1', 'anno_01', 'anno_02', 'anno_03', 'class_name_group', 'Sex_group', 'Sex_group1', 'anno_new', 'anno_new_class', 'new_class', 'Anno_Clusters', 'anno_0825'
    var: 'ENSid-0', 'n_cells-0', 'n_counts-0', 'mean_umi-0', 'ENSid-1', 'n_cells-1', 'n_counts-1', 'mean_umi-1', 'ENSid-10', 'n_cells-10', 'n_counts-10', 'mean_umi-10', 'ENSid-11', 'n_cells-11',

In [11]:
import numpy as np
import pandas as pd

adata = adata_raw

genes = ["C1qa", "C1qb", "C1qc", "Csf1r"]

# --- (1) 按 MouseID 做 pseudobulk ---
mouse_ids = adata.obs["MouseID"].unique()

bulk_counts = []
for mid in mouse_ids:
    m = adata.obs["MouseID"] == mid
    counts = np.asarray(adata[m].X.sum(axis=0)).ravel()
    bulk_counts.append(counts)

bulk_df = pd.DataFrame(
    bulk_counts,
    index=mouse_ids,
    columns=adata.var_names
)

# --- (2) CPM ---
cpm_df = bulk_df.div(bulk_df.sum(axis=1), axis=0) * 1e6

# --- (3) 基因存在性检查 + 取出这4个基因的 CPM ---
missing = [g for g in genes if g not in cpm_df.columns]
if missing:
    raise ValueError(f"These genes are not in adata.var_names: {missing}")

meta_mouse = (
    adata.obs[["MouseID", "Group", "Sex"]]
    .drop_duplicates(subset="MouseID")
    .set_index("MouseID")
)

plot_df = (
    cpm_df[genes]               # 这里一次取 4 列
    .join(meta_mouse)
    .reset_index()
)

plot_df["Group_Sex"] = plot_df["Group"].astype(str) + "_" + plot_df["Sex"].astype(str)

print(plot_df.head())


   index       C1qa       C1qb       C1qc      Csf1r Group Sex Group_Sex
0   3144  29.512059  18.764259  20.320549   9.886096   PLX   F     PLX_F
1   3145  41.197018  23.640551  24.647823  11.112354   PLX   F     PLX_F
2   3146  47.701068  29.902597  34.232629  13.876069   PLX   F     PLX_F
3   3147  31.311462  19.130936  19.498429   8.500271   PLX   F     PLX_F
4   3148  28.811712  14.642835  15.686376  10.443726   PLX   F     PLX_F


In [12]:
plot_df.to_csv('C1qa_C1qb_C1qc_Csf1r_CPM.csv')